### Reflexion
- Agent's self-correction or self-improvement, allowing agent to,
    - Iteratively refine its output
    - Adjust its approach based
       - Feedback
       - Internal critique
       - Comparison against desired criteria
- Agent evaluating its own work, output or internal state
- Using this evaluation to improve its performance or refine its response.
- Reflection introduces a feedback loop.

In [ ]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI

load_dotenv()

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

In [ ]:
class AgentState(TypedDict):
    input: str
    draft: str
    feedback: str
    iteration: int
    max_iterations: int
    final_output: str

In [ ]:
#Node 1: Generate summary
def generate_summary(input: AgentState):

    system_msg = f"""
    You are an expert in generating consise product summary for any idea or concept. 
    Write a concise summary for the following user input.
    """
    user_msg = f"""
    User idea or concept: {input['input']}
    """

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]

    response = model.invoke(messages)
    draft = response.content.strip()
    print("Draft Summary:", draft)
    return {"draft": draft}

In [ ]:
#Node 2: Critique draft summary
def critique_summary(input: AgentState):

    system_msg = f"""
    Critique the following product draft summary for clarity, conciseness, appeal and completeness. 
    Provide specific, actionable feedback to improve the draft.
    """
    user_msg = f"""
    Input idea or concept: {input['input']}
    Draft summary: {input['draft']}
    """

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]

    response = model.invoke(messages)
    critique = response.content.strip()
    print("Critique Summary:", critique)
    return {"feedback": critique}

In [ ]:
#Node 3: Refine summary
def refine_summary(input: AgentState):

    system_msg = f"""
    Refine the following product draft summary based on the critique feedback for given user input.
    """
    user_msg = f"""
    Input idea or concept: {input['input']}
    Draft summary: {input['draft']}
    Feedback: {input['feedback']}
    """

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]

    response = model.invoke(messages)
    refined_summary = response.content.strip()
    print("Refined Summary:", refined_summary)
    return {
        "refined_summary": refined_summary,
        "iteration": input["iteration"] + 1
    }

In [ ]:
from typing import Literal


def check_termination(input: AgentState) -> Literal["refine", "end"]:
    if input["iteration"] < input["max_iterations"]:
        return "refine"
    else:
        return "end"

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("generate", generate_summary)
workflow.add_node("refine", refine_summary)
workflow.add_node("critique", critique_summary)

workflow.set_entry_point("generate")
workflow.add_edge("generate", "critique")
workflow.add_conditional_edges(
    "critique",
    check_termination,
    {
        "refine": "refine",
        "end": END
    }
)

graph = workflow.compile()